Laster inn de nødvendige pakkene, og setter enn seed slik at modellen gir de samme tilfeldige tallene hver gang

In [86]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(100)

Leser inn dataen

In [87]:
s_df = pd.read_csv("raw_data/stations.csv")
t_df = pd.read_csv("raw_data/trips.csv")
w_df = pd.read_csv("raw_data/weather.csv")

Videre prøver jeg å rydde datasettet "stations" slik at den dataen jeg ikke trenger fjernes, og sorterer opp i datasettet

In [51]:
stasjoner = [
    "Møllendalsplass","Torgallmenningen","Grieghallen","Høyteknologisenteret",
    "Studentboligene","Akvariet","Damsgårdsveien 71","Dreggsallmenningen Sør",
    "Florida Bybanestopp"
]

# Filtrerer først, så dropper jeg ubrukte kolonner
s_df_clean = s_df[s_df["station"].isin(stasjoner)]
s_df_clean = s_df_clean.drop(columns=["longitude", "latitude", "skipped_updates"])

# Sørger for datetime i UTC
s_df_clean["timestamp"] = pd.to_datetime(s_df_clean["timestamp"], utc=True, errors="coerce")

# Sorterer og endrer index
s_sorted  = s_df_clean.sort_values(["station", "timestamp"])
s_indexed = s_sorted.set_index("timestamp")

# Resample til hver time pr stasjon, og fyll framover (LOCF) for relevante kolonner
cols = ["free_bikes", "free_spots"]  # tidsseriene vi vil resample
s_resampled = s_indexed.groupby("station")[cols].resample("h").ffill()

# Tilbake til vanlig index
s_hourly = s_resampled.reset_index()

s_hourly


,station,timestamp,free_bikes,free_spots
0,Akvariet,2024-04-10 21:00:00+00:00,NaN,NaN
1,Akvariet,2024-04-10 22:00:00+00:00,0.0,18.0
2,Akvariet,2024-04-10 23:00:00+00:00,0.0,18.0
3,Akvariet,2024-04-11 00:00:00+00:00,0.0,18.0
4,Akvariet,2024-04-11 01:00:00+00:00,0.0,18.0
...,...,...,...,...
83541,Torgallmenningen,2025-05-02 11:00:00+00:00,12.0,12.0
83542,Torgallmenningen,2025-05-02 12:00:00+00:00,12.0,12.0
83543,Torgallmenningen,2025-05-02 13:00:00+00:00,14.0,10.0
83544,Torgallmenningen,2025-05-02 14:00:00+00:00,15.0,9.0


Rydder "trips"

In [79]:
# dropper unødvendige kolonner, og setter tid til datetime utc
t_df_clean = t_df.drop(columns=["start_station_latitude", "start_station_longitude", "end_station_latitude", "end_station_longitude"])
t_df_clean["started_at"] = pd.to_datetime(t_df_clean["started_at"], utc=True, errors="coerce")
t_df_clean["ended_at"] = pd.to_datetime(t_df_clean["ended_at"], utc=True, errors="coerce")

# lager times-gulv
t_df_clean["started_h"] = t_df_clean["started_at"].dt.floor("h")
t_df_clean["ended_h"] = t_df_clean["ended_at"].dt.floor("h")

# Antall turer som starter per stasjon/time
starts = (t_df_clean.groupby(["start_station_name","started_h"])
            .size()
            .rename("trips_out")
            .reset_index()
            .rename(columns={"start_station_name":"station","started_h":"timestamp"}))

# turer som slutter per stasjon/time
ends = (t_df_clean.groupby(["end_station_name", "ended_h"]).size().rename("trips_in").reset_index().rename(columns={"end_station_name":"station","ended_h":"timestamp"}))

# slår sammen turer ut og inn
hourly_trips = pd.merge(starts, ends, on=["station", "timestamp"], how="outer").fillna(0)

hourly_trips

,station,timestamp,trips_out,trips_in
0,AdO arena,2023-01-02 05:00:00+00:00,0.0,1.0
1,AdO arena,2023-01-02 06:00:00+00:00,0.0,2.0
2,AdO arena,2023-01-02 07:00:00+00:00,2.0,2.0
3,AdO arena,2023-01-02 08:00:00+00:00,0.0,1.0
4,AdO arena,2023-01-02 11:00:00+00:00,0.0,1.0
...,...,...,...,...
752140,Øvre Korskirkeallmenning,2024-05-08 04:00:00+00:00,3.0,0.0
752141,Øvre Korskirkeallmenning,2024-05-08 05:00:00+00:00,1.0,1.0
752142,Øvre Korskirkeallmenning,2024-05-08 06:00:00+00:00,5.0,3.0
752143,Øvre Korskirkeallmenning,2024-05-08 07:00:00+00:00,1.0,2.0


Rydder "weather"

In [88]:
w_df["timestamp"] = pd.to_datetime(w_df["timestamp"], utc=True, errors="coerce")

w_df = w_df.set_index("timestamp")

# Times-oppløsning, gjennomsnitt per time
w_hourly = w_df.resample("h").mean().reset_index()

w_hourly.head()

,timestamp,temperature,precipitation,wind_speed
0,2023-12-31 23:00:00+00:00,3.0,0.0,17.9
1,2024-01-01 00:00:00+00:00,2.8,0.0,13.2
2,2024-01-01 01:00:00+00:00,2.6,0.0,10.5
3,2024-01-01 02:00:00+00:00,2.7,0.0,18.9
4,2024-01-01 03:00:00+00:00,2.1,0.0,18.4
